In [2]:
# Cell 1: Install all required packages for Gemini + FAISS + Flask
!pip install langchain-core langchain-google-genai langchain-community langchain-text-splitters faiss-cpu pymupdf flask pyngrok -q
print("✅ All dependencies installed successfully!")

✅ All dependencies installed successfully!


In [3]:
# Cell 2: Securely set Google API Key
import os
from getpass import getpass

# This will prompt you to safely paste your key without exposing it in plaintext
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API Key: ")

print("✅ Google API Key set successfully!")

Enter your Gemini API Key: ··········
✅ Google API Key set successfully!


In [4]:
# Cell 3: PDF Extraction Logic
import fitz  # PyMuPDF

def extract_text_from_bytes(pdf_bytes: bytes) -> str:
    """Extracts all text from raw PDF bytes."""
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text.strip()

print("✅ PDF Extraction utility defined!")

✅ PDF Extraction utility defined!


In [5]:
# Cell 4: RAG Pipeline (Gemini + FAISS)
import os
import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

# Verify key presence before initializing models
if "GOOGLE_API_KEY" not in os.environ:
    raise ValueError("❌ Gemini key not found! Please run Cell 2 again.")

# Setup production-grade models
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

# Structured Scoring Prompt Template
SCORING_PROMPT = PromptTemplate(
    input_variables=["resume_text", "job_description", "relevant_chunks"],
    template="""You are an expert HR recruiter and technical interviewer.
Analyze the resume against the job description and return a structured JSON response.

JOB DESCRIPTION:
{job_description}

RESUME TEXT:
{resume_text}

MOST RELEVANT RESUME SECTIONS (from semantic search):
{relevant_chunks}

Return ONLY a valid JSON object with this exact structure (do not include markdown formatting or backticks):
{{
  "fit_score": <integer 0-100>,
  "fit_level": "<Poor|Fair|Good|Excellent>",
  "matched_skills": ["skill1", "skill2"],
  "missing_skills": ["skill1", "skill2"],
  "strengths": ["strength1", "strength2"],
  "gaps": ["gap1", "gap2"],
  "recommendation": "<short hiring recommendation>",
  "summary": "<2-3 sentence candidate summary>"
}}"""
)

def build_faiss_index(text: str) -> FAISS:
    """Chunk text and build an in-memory FAISS vector store."""
    chunks = splitter.split_text(text)
    docs = [Document(page_content=chunk) for chunk in chunks]
    return FAISS.from_documents(docs, embeddings)

def screen_resume(resume_text: str, job_description: str) -> dict:
    """Main execution pipeline for evaluating resumes against job goals."""
    vectorstore = build_faiss_index(resume_text)

    # Context-aware semantic extraction
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    relevant_docs = retriever.invoke(job_description)
    relevant_chunks = "\n\n---\n\n".join([doc.page_content for doc in relevant_docs])

    # Run the execution chain
    chain = SCORING_PROMPT | llm
    response = chain.invoke({
        "resume_text": resume_text[:3000],  # safety token threshold limit
        "job_description": job_description,
        "relevant_chunks": relevant_chunks
    })

    raw_output = response.content

    # Safety normalization layers for varying response list types
    if isinstance(raw_output, list):
        raw_output = raw_output[0]
        if isinstance(raw_output, dict) and "text" in raw_output:
            raw_output = raw_output["text"]

    raw_output = str(raw_output).strip()

    # Safe structure parser processing
    try:
        return json.loads(raw_output)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"error": "Parsing failed", "raw": raw_output}

print("✅ Core RAG execution environment loaded!")

✅ Core RAG execution environment loaded!


/tmp/ipykernel_2655/3593178346.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [6]:
# Cell 5: Flask REST API Configurations
import traceback
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "message": "Gemini API screening pipeline online"})

@app.route("/screen", methods=["POST"])
def screen_text_endpoint():
    data = request.get_json() or {}
    if "resume_text" not in data or "job_description" not in data:
        return jsonify({"error": "Missing input context elements"}), 400
    try:
        return jsonify(screen_resume(data["resume_text"], data["job_description"]))
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500

@app.route("/screen-pdf", methods=["POST"])
def screen_pdf_endpoint():
    if "resume" not in request.files or "job_description" not in request.form:
        return jsonify({"error": "Incomplete multipart data payload received"}), 400
    try:
        pdf_bytes = request.files["resume"].read()
        resume_text = extract_text_from_bytes(pdf_bytes)
        job_description = request.form["job_description"]
        return jsonify(screen_resume(resume_text, job_description))
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500

print("✅ Flask application routes bound successfully!")

✅ Flask application routes bound successfully!


In [7]:
# Cell 6: Direct Frontend Routing Render Definition
from flask import render_template_string

frontend_html = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>AI Resume Screener</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: 'Segoe UI', sans-serif; background: #0f172a; color: #e2e8f0; min-height: 100vh; }
  header { background: linear-gradient(135deg, #1e293b, #0f172a); padding: 28px 40px; border-bottom: 1px solid #1e3a5f; }
  header h1 { font-size: 1.8rem; color: #38bdf8; }
  header p { color: #94a3b8; font-size: 0.9rem; margin-top: 4px; }
  .container { max-width: 900px; margin: 40px auto; padding: 0 20px; }
  .card { background: #1e293b; border-radius: 12px; padding: 28px; margin-bottom: 24px; border: 1px solid #1e3a5f; }
  .card h2 { color: #38bdf8; margin-bottom: 18px; font-size: 1.1rem; letter-spacing: 0.05em; text-transform: uppercase; }
  label { display: block; color: #94a3b8; font-size: 0.85rem; margin-bottom: 6px; margin-top: 14px; }
  textarea, input[type=file] { width: 100%; background: #0f172a; border: 1px solid #334155; color: #e2e8f0; border-radius: 8px; padding: 12px; font-size: 0.95rem; }
  textarea { resize: vertical; min-height: 140px; }
  input[type=file] { padding: 10px; cursor: pointer; }
  .tabs { display: flex; gap: 10px; margin-bottom: 20px; }
  .tab { padding: 8px 20px; border-radius: 6px; cursor: pointer; border: 1px solid #334155; background: #0f172a; color: #94a3b8; font-size: 0.9rem; }
  .tab.active { background: #38bdf8; color: #0f172a; font-weight: 700; border-color: #38bdf8; }
  .panel { display: none; }
  .panel.active { display: block; }
  button.submit { margin-top: 20px; width: 100%; padding: 14px; background: linear-gradient(135deg, #0ea5e9, #38bdf8); color: #0f172a; font-weight: 700; font-size: 1rem; border: none; border-radius: 8px; cursor: pointer; letter-spacing: 0.04em; }
  button.submit:hover { opacity: 0.9; }
  .result-card { background: #0f172a; border-radius: 10px; padding: 22px; border: 1px solid #1e3a5f; margin-bottom: 16px; }
  .score-row { display: flex; align-items: center; gap: 16px; margin-bottom: 20px; }
  .score-circle { width: 80px; height: 80px; border-radius: 50%; display: flex; align-items: center; justify-content: flex-direction: column; flex-direction: column; justify-content: center; font-size: 1.6rem; font-weight: 800; border: 4px solid #38bdf8; }
  .fit-label { font-size: 1.1rem; font-weight: 700; }
  .fit-excellent { color: #22c55e; } .fit-good { color: #38bdf8; } .fit-fair { color: #f59e0b; } .fit-poor { color: #ef4444; }
  .chip-row { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 8px; }
  .chip { padding: 4px 12px; border-radius: 20px; font-size: 0.8rem; font-weight: 600; }
  .chip.match { background: #052e16; color: #4ade80; border: 1px solid #166534; }
  .chip.miss { background: #450a0a; color: #f87171; border: 1px solid #7f1d1d; }
  .section-title { color: #94a3b8; font-size: 0.8rem; text-transform: uppercase; letter-spacing: 0.08em; margin-bottom: 8px; margin-top: 16px; }
  .summary { color: #cbd5e1; font-size: 0.95rem; line-height: 1.6; }
  .rec { background: #1e3a5f; padding: 12px 16px; border-radius: 8px; color: #7dd3fc; font-style: italic; margin-top: 8px; }
  #loading { display: none; text-align: center; padding: 30px; color: #38bdf8; font-size: 1.1rem; }
  #results { display: none; }
</style>
</head>
<body>
<header>
  <h1>🤖 AI Resume Screener</h1>
  <p>RAG-powered semantic matching · LangChain + FAISS + Gemini 3.5 Flash</p>
</header>
<div class="container">
  <div class="card">
    <h2>📋 Screen a Resume</h2>
    <div class="tabs">
      <div class="tab active" onclick="switchTab('text')">Paste Resume Text</div>
      <div class="tab" onclick="switchTab('pdf')">Upload PDF</div>
    </div>

    <div class="panel active" id="panel-text">
      <label>Resume Text</label>
      <textarea id="resume-text" placeholder="Paste the full resume here..."></textarea>
      <label>Job Description</label>
      <textarea id="jd-text" placeholder="Paste the job description here..."></textarea>
      <button class="submit" onclick="screenText()">🔍 Analyze Resume</button>
    </div>

    <div class="panel" id="panel-pdf">
      <label>Upload Resume PDF</label>
      <input type="file" id="resume-pdf" accept=".pdf" />
      <label>Job Description</label>
      <textarea id="jd-pdf" placeholder="Paste the job description here..."></textarea>
      <button class="submit" onclick="screenPDF()">🔍 Analyze Resume</button>
    </div>
  </div>

  <div id="loading">⏳ Analyzing resume with AI... please wait</div>
  <div id="results"></div>
</div>

<script>
function switchTab(tab) {
  document.querySelectorAll('.tab').forEach(t => t.classList.remove('active'));
  document.querySelectorAll('.panel').forEach(p => p.classList.remove('active'));
  event.target.classList.add('active');
  document.getElementById('panel-' + tab).classList.add('active');
}

async function screenText() {
  const resume_text = document.getElementById('resume-text').value.trim();
  const job_description = document.getElementById('jd-text').value.trim();
  if (!resume_text || !job_description) return alert('Both tracking contexts must contain input values.');

  showLoading();
  const res = await fetch('/screen', {
    method: 'POST', headers: {'Content-Type': 'application/json'},
    body: JSON.stringify({ resume_text, job_description })
  });
  renderResults(await res.json());
}

async function screenPDF() {
  const file = document.getElementById('resume-pdf').files[0];
  const jd = document.getElementById('jd-pdf').value.trim();
  if (!file || !jd) return alert('Please attach file document information and criteria data.');

  showLoading();
  const fd = new FormData();
  fd.append('resume', file);
  fd.append('job_description', jd);

  const res = await fetch('/screen-pdf', { method: 'POST', body: fd });
  renderResults(await res.json());
}

function showLoading() {
  document.getElementById('loading').style.display = 'block';
  document.getElementById('results').style.display = 'none';
}

function renderResults(d) {
  document.getElementById('loading').style.display = 'none';
  const r = document.getElementById('results');
  r.style.display = 'block';

  if (d.error) { r.innerHTML = `<div class="result-card" style="color:#f87171">❌ Error: ${d.error}</div>`; return; }

  const fitClass = { Excellent: 'fit-excellent', Good: 'fit-good', Fair: 'fit-fair', Poor: 'fit-poor' }[d.fit_level] || 'fit-fair';
  const matched = (d.matched_skills || []).map(s => `<span class="chip match">✓ ${s}</span>`).join('');
  const missing = (d.missing_skills || []).map(s => `<span class="chip miss">✗ ${s}</span>`).join('');
  const strengths = (d.strengths || []).map(s => `<li style="color:#94a3b8;margin:4px 0">• ${s}</li>`).join('');
  const gaps = (d.gaps || []).map(s => `<li style="color:#94a3b8;margin:4px 0">• ${s}</li>`).join('');

  r.innerHTML = `
    <div class="card">
      <h2>📊 Screening Results</h2>
      <div class="result-card">
        <div class="score-row">
          <div class="score-circle">${d.fit_score}</div>
          <div>
            <div class="section-title">Fit Score / 100</div>
            <div class="fit-label ${fitClass}">${d.fit_level} Match</div>
          </div>
        </div>
        <div class="section-title">Matched Skills</div>
        <div class="chip-row">${matched || 'None detected'}</div>
        <div class="section-title">Missing Skills</div>
        <div class="chip-row">${missing || 'None detected'}</div>
        <div class="section-title">Strengths</div>
        <ul style="list-style:none">${strengths}</ul>
        <div class="section-title">Gaps</div>
        <ul style="list-style:none">${gaps}</ul>
        <div class="section-title">Summary</div>
        <div class="summary">${d.summary}</div>
        <div class="section-title">Recommendation</div>
        <div class="rec">💼 ${d.recommendation}</div>
      </div>
    </div>`;
}
</script>
</body>
</html>
"""

@app.route("/")
def index():
    return render_template_string(frontend_html)

print("✅ UI Rendering parameters mapped completely!")

✅ UI Rendering parameters mapped completely!


In [13]:
# Cell 8: Launch Flask with ngrok tunnel
from pyngrok import ngrok
import threading
import time

# ── Set your ngrok auth token ──────────────────────────────────
# Paste your real ngrok token inside the quotes below!
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"

if NGROK_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    raise ValueError("❌ Please paste your real ngrok token in line 7!")

ngrok.set_auth_token(NGROK_TOKEN)

# ── Kill any existing tunnels to prevent errors ────────────────
ngrok.kill()

# ── Start Flask in a background thread ──────────────────────────
def run_flask():
    # We run on port 5000, which is standard for Flask
    app.run(host="0.0.0.0", port=5000, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

# ── Create ngrok public URL ────────────────────────────────────
time.sleep(2)  # Wait just a moment for Flask to spin up
public_url = ngrok.connect(5000)

print("=" * 65)
print(f"🎉 YOUR AI RESUME SCREENER IS LIVE!")
print(f"👉 CLICK THIS LINK TO OPEN YOUR APP: {public_url}")
print("=" * 65)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🎉 YOUR AI RESUME SCREENER IS LIVE!
👉 CLICK THIS LINK TO OPEN YOUR APP: NgrokTunnel: "https://saline-drinking-hedging.ngrok-free.dev" -> "http://localhost:5000"
